# Experiment 6 — RSA Algorithm
Key generation · Encryption · Decryption

## Math Helpers — Extended GCD & Miller-Rabin

In [ ]:
import random, math, secrets

def ext_gcd(a,b):
    if a==0: return b,0,1
    g,x,y=ext_gcd(b%a,a); return g,y-(b//a)*x,x

def mod_inv(a,m):
    g,x,_=ext_gcd(a%m,m)
    if g!=1: raise ValueError(f"No inverse: gcd({a},{m})={g}")
    return x%m

def miller_rabin(n,k=10):
    if n<2: return False
    if n in(2,3): return True
    if n%2==0: return False
    r,d=0,n-1
    while d%2==0: r+=1; d//=2
    for _ in range(k):
        a=random.randrange(2,n-1); x=pow(a,d,n)
        if x in(1,n-1): continue
        for _ in range(r-1):
            x=pow(x,2,n)
            if x==n-1: break
        else: return False
    return True

def gen_prime(bits):
    while True:
        n=secrets.randbits(bits); n|=(1<<(bits-1))|1
        if miller_rabin(n): return n

print("Helpers defined ✓")


## Key Generation

In [ ]:
def rsa_keygen(bits=512):
    half=bits//2
    while True:
        p=gen_prime(half); q=gen_prime(half)
        if p==q: continue
        n=p*q; phi=(p-1)*(q-1); e=65537
        if math.gcd(e,phi)==1: break
    d=mod_inv(e,phi)
    return (e,n),(d,n),p,q

pub,priv,p,q=rsa_keygen(512)
e,n=pub; d,_=priv
print(f"p = {p}")
print(f"q = {q}")
print(f"n = {n}")
print(f"e = {e}  (Fermat prime)")
print(f"d = {d}")
phi=(p-1)*(q-1)
assert (e*d)%phi==1
print(f"e×d mod φ(n) = {(e*d)%phi}  ✓")


## Encrypt / Decrypt

In [ ]:
def rsa_enc(m,pub): return pow(m,pub[0],pub[1])
def rsa_dec(c,priv): return pow(c,priv[0],priv[1])

def txt2int(t): return int.from_bytes(t.encode(),'big')
def int2txt(n): return n.to_bytes((n.bit_length()+7)//8,'big').decode()

# Integer message
m=42; ct=rsa_enc(m,pub); dt=rsa_dec(ct,priv)
print(f"Plaintext  : {m}")
print(f"Ciphertext : {ct}")
print(f"Decrypted  : {dt}")
assert dt==m
print("✓ Integer roundtrip verified")
print()

# Text message
msg="RSA Demo"; mi=txt2int(msg); ci=rsa_enc(mi,pub); di=rsa_dec(ci,pub)
# Note: decrypt with private key
di=rsa_dec(ci,priv)
print(f"Message   : '{msg}'")
print(f"Encrypted : {rsa_enc(mi,pub)}")
print(f"Decrypted : '{int2txt(di)}'")
assert int2txt(di)==msg
print("✓ Text roundtrip verified")


## Security Properties

In [ ]:
print(f"Key size         : 512-bit modulus")
print(f"n bit length     : {n.bit_length()} bits")
print(f"e = {e} (standard Fermat prime, fast public-key ops)")
print()
print("Security relies on hardness of integer factorisation.")
print("RSA-2048 (2048-bit n) is current recommended minimum.")
print()
print("Encryption: C = M^e mod n")
print("Decryption: M = C^d mod n")
print("Signing   : S = M^d mod n  (verify: M = S^e mod n)")
